In [1]:
from pathlib import Path

In [2]:
#testing data path
series_folder = Path(r"E:\PROJECTS\EM\zeinab\ATLAS\data\zeinab")
series_folder = Path(r"/home/xgupke/Documents/data/ZEINAB-ROI-w1-20nm-bsd")

series_list = []
for folder in series_folder.iterdir():  # Iterate over all items in the folder
    if folder.is_dir() and folder.name.startswith("S_"):
        tif_files = list(folder.glob("*.tif"))
        if tif_files:  # Check if the list is not empty
            print(f"Found series folder: {folder.name} (contains {len(tif_files)} .tif files)")
            series_list.append(folder)

print(f"\nFound {len(series_list)} series.")

Found series folder: S_009_183996284 (contains 3 .tif files)
Found series folder: S_003_1082014046 (contains 3 .tif files)
Found series folder: S_004_520641757 (contains 3 .tif files)
Found series folder: S_007_219502197 (contains 3 .tif files)
Found series folder: S_006_1441661872 (contains 3 .tif files)
Found series folder: S_005_1107052629 (contains 3 .tif files)
Found series folder: S_020_876249941 (contains 3 .tif files)
Found series folder: S_010_1018560366 (contains 3 .tif files)
Found series folder: S_008_564872779 (contains 3 .tif files)
Found series folder: S_017_314689960 (contains 3 .tif files)
Found series folder: S_012_343837657 (contains 3 .tif files)
Found series folder: S_019_971884407 (contains 3 .tif files)
Found series folder: S_014_2134637132 (contains 3 .tif files)
Found series folder: S_021_2072832173 (contains 3 .tif files)
Found series folder: S_016_1415363463 (contains 3 .tif files)
Found series folder: S_011_2059090218 (contains 3 .tif files)
Found series fol

In [3]:
import json

import numpy as np
import tifffile as tiff
from scipy.sparse import csr_matrix
from scipy.sparse.csgraph import minimum_spanning_tree

from atlas.image_analysis import calculate_mask_roi, mask_low_and_saturation

from atlas.io import extract_s_number
from atlas.stitching import (
    stitch_ATLAS_tiles,
    add_tile_overlap_columns,
    apply_transforms_and_stitch,
    build_adjacency_matrix_from_costs,
    build_transform_dict_from_mst,
    get_tiles_dataframe,
    match_tiles,
)

In [5]:
buffer_in_microns = 1

max_shift_in_pixles = 200

# iterate over all folders in the series
for raw_data_folder in series_list:
    # Iterate over all items in the folder
    for file in raw_data_folder.iterdir():  
        # Check if it's a file with the desired extension
        if file.is_file() and file.suffix == ".ve-mif":
            print(f"File with '.ve-mif' extension found: {file.name}")
            mif_file = file
            # use naming convention to find the first tif
            first_tif_path = next(raw_data_folder.glob("*.tif"))
            print(first_tif_path)
            extracted_number = extract_s_number(first_tif_path)
            # Define the output file path
            output_tif_path = raw_data_folder.parent.joinpath(f"stitched_image_S_{extracted_number}.tiff")
            output_cc_path = raw_data_folder.parent.joinpath(f"phaseCC_stitching_S_{extracted_number}.csv")
            output_jason_path = raw_data_folder.parent.joinpath(f"transforms_S_{extracted_number}.json")

            if output_tif_path.exists():
                print(f"✅ Skipping: {output_tif_path.name} already exists.")
            else:
                print(f"🔄 Stitching image for S_{extracted_number}...")

                try:
                    stitched_img, mif_tile_df, transform_dict = stitch_ATLAS_tiles(
                        mif_file,
                        buffer_microns=buffer_in_microns,
                        max_shift_pixels=max_shift_in_pixles,
                    )

                    # Save the stitched image as a TIFF file
                    tiff.imwrite(output_tif_path, np.flipud(stitched_img))
                    

                    mif_tile_df.to_csv(output_cc_path, index=False)

                    # Convert NumPy arrays to lists for JSON compatibility
                    json_ready_dict = {k: v.tolist() for k, v in transform_dict.items()}

                    # Save to JSON file
                    with open(output_jason_path, "w") as f:
                        json.dump(json_ready_dict, f, indent=2)
                        
                except Exception as e:
                    # Handle any error
                    print(f"Unexpected error with item {file}: {e}")
            

File with '.ve-mif' extension found: MosaicInfo_S_009_183996284.ve-mif
/home/xgupke/Documents/data/ZEINAB-ROI-w1-20nm-bsd/S_009_183996284/Tile_r3-c1_S_009_183996284.tif
🔄 Stitching image for S_9...




Processing reference tile 0...

Comparing reference 0 to tile 0...
Overlap %: 0
Too little overlap: assigning cost=1.0 and shift=[0,0]

Comparing reference 0 to tile 1...
Overlap %: 6
overlap img0 x: 0-7000, y 0-419
overlap img1 x: 0-7000, y 6581-7000
Detected pixel offset (row, col): [-8. -9.]

Comparing reference 0 to tile 2...
Overlap %: 0
Too little overlap: assigning cost=1.0 and shift=[0,0]

Processing reference tile 1...

Comparing reference 1 to tile 0...
Overlap %: 6
overlap img0 x: 0-7000, y 6581-7000
overlap img1 x: 0-7000, y 0-419
Detected pixel offset (row, col): [8. 9.]

Comparing reference 1 to tile 1...
Overlap %: 0
Too little overlap: assigning cost=1.0 and shift=[0,0]

Comparing reference 1 to tile 2...
Overlap %: 6
overlap img0 x: 0-7000, y 0-420
overlap img1 x: 0-7000